# Facilitator Answer Key — v2 Aligned Workflow

## Activity 2

In [ ]:
measurements=["red","green","blue","nir08","swir16","scl"]
mask_flags=[1,3,9,10]

## Activity 3

In [ ]:
aoi_gdf = gpd.read_file(AOI_PATH)
gdf = gpd.read_file(TRAINING_PATH, bbox=tuple(bbox))

## Activity 4

In [ ]:
datetime="2018-03/2018-07"
# collection: sentinel-2-c1-l2a
# source example cloud threshold: 25
# load bands: red, green, blue, nir08, swir16, scl

## Activity 5

In [ ]:
mask_flags=[1,3,9,10]
cloud_mask = ~data.scl.isin(mask_flags)
masked = data.where(cloud_mask)
non_zero = masked.where(masked != 0)
scaled = non_zero * 0.0001
scaled = scaled.clip(0,1)
scaled["ndvi"] = (scaled.nir08-scaled.red)/(scaled.nir08+scaled.red)
median = scaled.median("time").compute()

## Activity 6

In [ ]:
training = gdf.to_crs(median.odc.geobox.crs)
training_da = training.assign(x=training.geometry.x,y=training.geometry.y).to_xarray()
training_values=(median.sel(training_da[["x","y"]],method="nearest").squeeze().compute().to_pandas())
training_array=pd.concat([training["randomforest"],training_values],axis=1)
training_array=training_array.drop(columns=["y","x","spatial_ref"]).dropna()

## Activity 7

In [ ]:
classes=np.array(training_array)[:,0]
observations=np.array(training_array)[:,1:]
classifier=RandomForestClassifier(random_state=42)
model=classifier.fit(observations,classes)

## Activity 8

In [ ]:
stacked_arrays=median.to_array().stack(dims=["y","x"]).transpose()
predicted=model.predict(stacked_arrays)
array=predicted.reshape(len(median.y),len(median.x))
predicted_da=xr.DataArray(array,coords={"y":median.y,"x":median.x},dims=["y","x"]).astype("float32")
predicted_da.odc.write_cog("../outputs/cordia_prediction.tif")

## Activity 9

In [ ]:
cordia_pixels=int((predicted_da==CORDIA_CLASS).sum().item())
cordia_area_m2=cordia_pixels*pixel_area_m2
cordia_area_ha=cordia_area_m2/10000
valid_pixels=int(predicted_da.notnull().sum().item())
cordia_percent=(cordia_pixels/valid_pixels)*100

# Added Facilitator Notes — Workbook 4 Monthly Sentinel-2 Visuals

These notes correspond to the monthly visual-inspection activities added to Workbook 4. The exact "best" month is data-dependent, so participants should support their answers using the imagery they actually retrieve.

## Workbook 4 — Activity 4.7: Individual acquisitions

**Expected observation:** Participants should notice that individual Sentinel-2 acquisitions can differ substantially in cloud cover, haze, visible land surface, and vegetation appearance.

**Facilitator prompt:** Ask participants why an image with a low catalogue-level cloud percentage can still contain cloud over the specific workshop AOI.

**Expected answer:** `eo:cloud_cover` describes cloudiness for the Sentinel-2 item/tile rather than guaranteeing that the workshop AOI itself is cloud-free.

## Workbook 4 — Challenge 4.10: Compare the months

There is **no single fixed month answer** because the result depends on the selected date range and available Sentinel-2 observations.

A strong participant answer should identify:

- the month with the least visible cloud/haze over the AOI;
- months where vegetation/canopy appearance differs;
- whether the target species may be more distinguishable during its flowering/seasonal period;
- whether enough usable imagery exists for that month.

**Suggested interpretation:**

> The most useful month is not automatically the month with the lowest cloud. Ideally, the selected period combines relatively clear imagery with the season when the target invasive species has a distinctive canopy, flowering, or spectral response.

**Example only — participants must replace this with their actual observations:**

| Month | Cloud condition | Vegetation appearance | Potential usefulness |
|---|---|---|---|
| March | High cloud | Canopy partly obscured | Low–Moderate |
| April | Moderate cloud | Vegetation more visible | Moderate |
| May | Low cloud | Clearer canopy differences | High |

**Teaching point:** This exercise provides the ecological justification for experimenting with the STAC `datetime` range rather than choosing dates arbitrarily.

# Added Facilitator Notes — Workbook 5 Cloud-Masking Visuals

## Workbook 5 — Activity 5.6: Raw vs cloud-masked acquisition

**Expected observation:** Pixels belonging to the selected SCL mask classes should disappear from the masked image and become missing/NoData.

The workshop mask is:

- `1` — defective pixels;
- `3` — cloud shadow;
- `9` — high-confidence cloud;
- `10` — thin cirrus.

Participants should recognise that masking improves data quality but does **not** magically reconstruct the land surface beneath clouds. It creates gaps where unreliable observations were removed.

## Workbook 5 — Activity 5.8: Monthly imagery after masking

**Expected observation:** Compared with Workbook 4, cloud-contaminated areas should be reduced after applying the SCL mask. However, some months may contain substantial gaps if few clear observations are available.

**Facilitator prompt:** Ask: "Is the image necessarily better simply because cloud pixels have disappeared?"

**Expected answer:** It is cleaner for analysis, but missing pixels may still make a single date/month unsuitable for wall-to-wall classification.

## Workbook 5 — Challenge 5.9: Before vs after masking

A strong response should explain:

> Before masking, clouds, cloud shadows, cirrus and defective pixels can contaminate the spectral information used by the classifier. After applying the SCL mask, those unreliable pixels are excluded, producing cleaner observations but leaving missing areas.

Example comparison:

| Stage | Expected observation |
|---|---|
| Before masking | Cloud/shadow may obscure vegetation and alter spectral values |
| After masking | Problem pixels are removed but gaps/NoData remain |
| Temporal median | Multiple observations are combined to produce a more stable composite |

**Why use a temporal median?**

> The temporal median combines multiple observations so that extreme or temporary values have less influence and valid observations from different dates contribute to a more stable representation of the landscape. This can reduce residual cloud/shadow effects and short-term variability before classification.

**Important nuance:** A temporal median does not recover information where every observation is invalid or masked.

## Connecting Workbooks 4 and 5 to invasive-species detection

Use this sequence during discussion:

**Raw monthly imagery → identify seasonal/cloud differences → SCL masking → compare before/after → scale reflectance → calculate NDVI → temporal median → extract training values → Random Forest**

The key learning message is:

> We are not preprocessing imagery simply because Python requires it. Each step is intended to improve the quality and consistency of the information presented to the classifier.

## Troubleshooting notes

**Selected month gives a KeyError:**  
The chosen month is probably not present in `monthly_raw.month.values` or `monthly_masked.month.values`. Ask participants to choose one of the available month numbers printed by the previous cell.

**Only one acquisition is returned:**  
The date range may be too narrow or the cloud threshold too restrictive. Review `datetime` and the STAC cloud-cover filter.

**Map looks very dark or bright:**  
Adjust the RGB display range (`vmin` / `vmax`). This changes visualisation only; it does not alter the underlying Sentinel-2 data.

**Large gaps remain after masking:**  
This can be a legitimate result when cloud/shadow affects many observations. It reinforces the reason for using multiple dates and a temporal composite.

**A cloud still appears after masking:**  
SCL classification is not perfect. Cloud masking reduces contamination but should not be treated as flawless ground truth.